# 38 - Prefix-only versus reduced-strength P&P, worker 0/2

This is shard 0 of 2 over the same 220 held-out LIBERO-PRO identities as the suffix-sensitivity pilot: 11 suites x 10 tasks x init indices 10 and 11. The two short-asset milk suites do not provide these initialization indices and are excluded.

Every identity runs two paired arms. **Prefix-only** updates latent action positions 0-9 during each inner K-step P&P loop while positions 10-49 retain their ordinary sampler state. **Reduced-strength** updates the full 50-position latent but moves only `beta=0.5` toward each inner predict/perturb proposal. Both use K=5, Euler steps `(3,4)`, refine-last, and execute 10 actions.

Each worker runs 110 identities x 2 arms = 220 rollouts. Every 20 rollouts, the notebook prints balanced per-suite SR plus U10, U20, and full-chunk uncertainty means for both arms. Worker 0 may set `EPISODE_LIMIT = 1` for a two-rollout smoke test, then restore `None`; those rows resume safely.

## 1. Setup a fresh GPU runtime

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration and resumable paired collection

In [ ]:
from pathlib import Path
from google.colab import drive
from pnp.config import PI05_REPO_ID
from pnp.diversity import (SOURCE_PREFIX_STRENGTH_EXPERIMENT,
    load_bootstrap_manifest, run_source_prefix_strength_worker)

drive.mount('/content/drive')

EPISODE_INDICES = (10, 11)
INNER_STRENGTH = 0.5
SHARD_COUNT = 2
SHARD_INDEX = 0
EPISODE_LIMIT = None  # optional smoke: set 1 once, then restore None
EXPERIMENT = SOURCE_PREFIX_STRENGTH_EXPERIMENT
MANIFEST_PATH = Path(
    '/content/drive/MyDrive/pnp_diversity_v2/bootstrap_manifest_finetuned_v2.json')
manifest = load_bootstrap_manifest(MANIFEST_PATH)
assert manifest['source_model'] == PI05_REPO_ID, manifest['source_model']
SOURCE_MODEL_REVISION = manifest['source_model_revision']
assert SOURCE_MODEL_REVISION, 'v2 manifest is missing source_model_revision'

print({'experiment': EXPERIMENT, 'arms': ['prefix-only', 'inner-strength beta=0.5'],
       'episode_indices': EPISODE_INDICES, 'full_subset_identities': 220,
       'identities_in_this_shard': 110, 'rollouts_in_this_shard': 220,
       'inner_strength': INNER_STRENGTH,
       'shard_count': SHARD_COUNT, 'shard_index': SHARD_INDEX,
       'episode_limit': EPISODE_LIMIT, 'manifest_hash': manifest['manifest_hash'],
       'source_model_revision': SOURCE_MODEL_REVISION})
run_source_prefix_strength_worker(
    episode_indices=EPISODE_INDICES, episode_limit=EPISODE_LIMIT,
    inner_strength=INNER_STRENGTH,
    shard_count=SHARD_COUNT, shard_index=SHARD_INDEX,
    manifest_hash=manifest['manifest_hash'],
    source_model_revision=SOURCE_MODEL_REVISION, experiment=EXPERIMENT)